In [1]:
import json
import hashlib
import sys
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "qwen3:30b-a3b-instruct-2507-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"

# 1. Leer archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
configuracion = json.loads(bytes_config)

# 2. Verificar integridad
sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == configuracion["cases_sha256"]

assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert configuracion["prompt_version"] == "V3"
assert configuracion["api_url"] == API_LOCAL + "/api/chat"

# 3. Conectar al servidor local
cliente_local = build_opener(ProxyHandler({}))

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as f:
    version_ollama = json.load(f)["version"]

with cliente_local.open(API_LOCAL + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    (m for m in modelos if m["name"] == MODELO),
    None
)

assert modelo_info is not None, "Modelo no instalado."

# 4. Consultar características reales
peticion = Request(
    API_LOCAL + "/api/show",
    data=json.dumps({"model": MODELO}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with cliente_local.open(peticion, timeout=15) as f:
    detalles = json.load(f)

# 5. Mostrar resultados
print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Modelo:", MODELO)
print("Identificador:", modelo_info["digest"][:12])
print("Casos verificados:", len(casos))
print("SHA256:", sha_casos)
print("Prompt:", configuracion["prompt_version"])
print("Capacidades:", detalles.get("capabilities"))
print("Thinking:", detalles.get("thinking"))
print("Parámetros:", detalles.get("parameters"))

print("\nArchivos y conexión: correctos")


Python: 3.11.16
Ollama: 0.40.13
Modelo: qwen3:30b-a3b-instruct-2507-q4_K_M
Identificador: 19e422b02313
Casos verificados: 6
SHA256: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
Prompt: V3
Capacidades: ['tools', 'thinking', 'completion']
Thinking: None
Parámetros: temperature                    0.7
top_k                          20
top_p                          0.8
repeat_penalty                 1
stop                           "<|im_start|>"
stop                           "<|im_end|>"

Archivos y conexión: correctos


In [2]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request

ARCHIVO = "resultados.jsonl"
CASO_ID = "DIAG-Q3-30B-A3B-01"
CONFIG_MODELO = "qwen3_30b_a3b_instruct_directa_v1"

# 1. Caso diagnóstico común
contenido = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# 2. Recuperar configuración compartida V3
solicitud = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud["model"] = MODELO

# Variante Instruct-2507: solo respuesta directa
solicitud.pop("think", None)

solicitud["options"] = {
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "min_p": 0,
    "repeat_penalty": 1
}

solicitud["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id=CASO_ID,
            contenido=contenido
        )
    }
]

# 3. Verificar modelo y archivos comunes
assert MODELO == "qwen3:30b-a3b-instruct-2507-q4_K_M"
assert configuracion["prompt_version"] == "V3"
assert sha_casos == configuracion["cases_sha256"]
assert modelo_info["digest"].startswith("19e422b02313")
assert "completion" in detalles.get("capabilities", [])
assert "think" not in solicitud

# Evitar diagnósticos duplicados
try:
    with open(ARCHIVO, encoding="utf-8") as f:
        anteriores = [
            json.loads(line)
            for line in f if line.strip()
        ]
except FileNotFoundError:
    anteriores = []

assert not any(
    r.get("case_id") == CASO_ID
    and r.get("configuracion_modelo") == CONFIG_MODELO
    for r in anteriores
), "El diagnóstico ya está registrado."

# 4. Preparar registro experimental
registro = {
    "execution_id": str(uuid4()),
    "case_id": CASO_ID,
    "grupo": "diagnostico",
    "configuracion_modelo": CONFIG_MODELO,
    "prompt_version": "V3",
    "modelo_digest": modelo_info["digest"],
    "ollama_version": version_ollama,
    "sha256_casos": sha_casos,
    "sha256_configuracion": sha_config,
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

# 5. Ejecutar diagnóstico en Ollama
try:
    peticion = Request(
        configuracion["api_url"],
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente_local.open(peticion, timeout=900) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})
    contenido_final = mensaje.get("content", "").strip()

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif mensaje.get("thinking"):
        registro["estado"] = "revisar_modo"

    elif (
        respuesta.get("done_reason") == "stop"
        and contenido_final
    ):
        registro["estado"] = "ok"

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 6. Guardar resultado sin sobrescribir
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(
        json.dumps(registro, ensure_ascii=False) + "\n"
    )

# 7. Mostrar métricas y respuesta
print("Caso:", CASO_ID)
print("Configuración:", CONFIG_MODELO)
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]
    mensaje = r.get("message", {})

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        mensaje.get("thinking")
    ))

    for campo in (
        "total_duration",
        "load_duration",
        "prompt_eval_duration",
        "eval_duration"
    ):
        valor = r.get(campo)

        if valor is not None:
            print(campo, round(valor / 1e9, 3), "s")

    print("\nRESPUESTA FINAL:\n")
    print(mensaje.get("content", ""))

else:
    print("ERROR:", registro.get("error"))

print("\nRegistro guardado en resultados.jsonl")


Caso: DIAG-Q3-30B-A3B-01
Configuración: qwen3_30b_a3b_instruct_directa_v1
Estado: ok
Tiempo cliente: 33.802 s
Tokens entrada: 426
Tokens generados: 492
Terminación: stop
Razonamiento separado: False
total_duration 33.8 s
load_duration 17.239 s
prompt_eval_duration 0.792 s
eval_duration 15.758 s

RESPUESTA FINAL:

**Evidencia**  
[E1] Autor X ha publicado un artículo centrado en bases de datos académicas.  
[E2] Autor Y ha publicado un artículo enfocado en recuperación de información.  

**Interpretación**  
Las áreas temáticas de los artículos de Autor X y Autor Y presentan un fuerte solapamiento en el ámbito de la informática académica y el manejo de información científica. La recuperación de información (E2) es un campo que depende, en gran medida, de la estructura y calidad de las bases de datos académicas (E1). Esta relación funcional entre ambos temas —donde la recuperación de información requiere acceso a bases de datos bien organizadas— sugiere una posible convergencia de intere

In [3]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "qwen3:30b-a3b-instruct-2507-q4_K_M"
API = "http://127.0.0.1:11434"
ARCHIVO = "resultados.jsonl"
CONFIG_MODELO = "qwen3_30b_a3b_instruct_directa_v1"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]

assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert config["prompt_version"] == "V3"
assert config["api_url"] == API + "/api/chat"

# 2. Recuperar diagnóstico exitoso
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line)
        for line in f if line.strip()
    ]

diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-Q3-30B-A3B-01"
    and r["configuracion_modelo"] == CONFIG_MODELO
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["model"] == MODELO
assert "think" not in diagnostico["solicitud"]

assert diagnostico["solicitud"]["options"] == {
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "min_p": 0,
    "repeat_penalty": 1
}

assert diagnostico["sha256_casos"] == sha_casos
assert diagnostico["sha256_configuracion"] == sha_config

assert diagnostico["solicitud"]["messages"][0]["content"] == (
    config["system_prompt"]
)

# 3. Comprobar identidad del modelo instalado
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# Evitar duplicar los casos ya registrados
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == CONFIG_MODELO
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado", flush=True)
            continue

        # Recuperar solicitud exacta del diagnóstico
        solicitud = copy.deepcopy(
            diagnostico["solicitud"]
        )

        # Sustituir únicamente el caso y sus evidencias
        solicitud["messages"][1]["content"] = (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        # Preparar registro experimental
        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": CONFIG_MODELO,
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        print("Ejecutando:", case_id, flush=True)
        inicio = time.perf_counter()

        # 5. Consultar API local de Ollama
        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=900) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})
            contenido_final = mensaje.get("content", "").strip()

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif mensaje.get("thinking"):
                registro["estado"] = "revisar_modo"

            elif (
                respuesta.get("done_reason") == "stop"
                and contenido_final
            ):
                registro["estado"] = "ok"

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # 6. Guardar cada ejecución inmediatamente
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s",
            flush=True
        )

print("\nArchivo actualizado:", ARCHIVO)


Ejecutando: S-005
S-005 | ok | 697 tokens | 21.249 s
Ejecutando: S-006
S-006 | ok | 564 tokens | 16.87 s
Ejecutando: S-007
S-007 | ok | 715 tokens | 20.626 s
Ejecutando: S-008
S-008 | ok | 540 tokens | 16.392 s
Ejecutando: S-009
S-009 | ok | 692 tokens | 20.775 s
Ejecutando: S-010
S-010 | ok | 644 tokens | 18.692 s

Archivo actualizado: resultados.jsonl
